# Chronos-2 experiment — zero-shot forecast of the Oct–Dec festive window

**Purpose.** Find out whether a pretrained foundation model (Chronos-2, 120M parameters, no training on your data) gets anywhere near your TFT on this problem, and which covariates help it.
It reads `series_cache/`, `shared_calendar.parquet` and `column_roles.json`. `series_cache/` is normally written by the modelling notebook (Step 2). If it is not found, **Section 0 builds it** from the `local_train_data/chunk_*.parquet` files that the data-preparation notebook (Step 1) wrote. Nothing is pulled from Snowflake until the optional last section.

**Folder.** Everything is read from `BASE` (set in the first code cell). If you run this notebook outside your TFT project folder, set `BASE` to that folder's full path.

**Why Chronos-2 and not the older Chronos / Chronos-Bolt.** Only Chronos-2 takes known-future covariates natively and can forecast 92 days in one call (its limit is 1,024). The older models stop at 64 steps and have no covariate input. Install: `pip install chronos-forecasting` (version 2.x).

**What is compared (all on the same series, same 92-day window)**

| Name | What it is |
|---|---|
| `chronos_mean` | Chronos-2 quantile forecast turned into a mean (see Section 4) |
| `chronos_median` | Chronos-2 `predictions` column (this is the **median**, not the mean) |
| `fasn` | Festival-aligned seasonal naive: last year's sales shifted so Navratri lines up with this year's Navratri |
| `trailing` | Mean of the last 91 days, flat |

**Two modes**
1. **Backtest (Sections 5–6).** Cutoff 2025-09-30, forecast 2025-10-01 → 2025-12-31, scored against what really happened. This is a fair test of Chronos because it never trained on your data. **It cannot be used to compare against the TFT**: the TFT trained on those very days.
2. **Live (Section 7).** Cutoff 2026-09-30, forecast Oct–Dec 2026. Section 8 scores Chronos and the TFT against real Oct–Nov 2026 sales if you export them. That is the only fair head-to-head with the TFT.

**Things that can mislead you**
- Chronos returns quantiles. For sparse counts (most series-days are 0) the median is almost always 0, so **totals from the median are far too low**. Judge totals and shape on `chronos_mean`.
- A high WAPE on single series-days says little. Read the aggregated levels too (family, dealer-week, national).
- Quantiles can cross and go below 0. Section 4 repairs both before any metric is computed.

In [ ]:
import os, json, time, warnings, glob, gc, shutil, hashlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from chronos import Chronos2Pipeline

BASE      = os.getcwd()     # folder holding column_roles.json, shared_calendar.parquet, series_cache/ and local_train_data/
# BASE  = r"C:\Users\<you>\Desktop\<your TFT project folder>"   # <- use this line if the notebook runs from another folder
CACHE_DIR = os.path.join(BASE, "series_cache")
with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)

time_col, group_col, target_col = ROLES["time_col"], ROLES["group_col"], ROLES["target_col"]
TRAIN_START, TRAIN_END = pd.Timestamp(ROLES["train_start"]), pd.Timestamp(ROLES["train_end"])
VAL_END                = pd.Timestamp(ROLES["val_end"])
FORECAST_START, FORECAST_END = pd.Timestamp(ROLES["forecast_start"]), pd.Timestamp(ROLES["forecast_end"])
HORIZON   = (FORECAST_END - FORECAST_START).days + 1            # 92
DAYS_FULL = pd.date_range(TRAIN_START, FORECAST_END, freq="D")   # calendar / spend run over this span
N_FULL, N_HIST = len(DAYS_FULL), (VAL_END - TRAIN_START).days + 1

# ------------------------------- settings you may change -----------------------------------
MODEL_ID        = "amazon/chronos-2"      # or "autogluon/chronos-2-small" (28M, faster, usually a bit worse)
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_N        = 3000                    # series used in the backtest; None = all (slow, do it last)
SEED            = 0
BACKTEST_CUTOFF = pd.Timestamp("2025-09-30")   # last day of context; forecast = next 92 days
RUN_SETS        = ["none", "calendar", "calendar+spend"]   # add "all_flags+spend" to give it everything the TFT gets
BLOCK           = 500                     # series per predict_df call (RAM)
BATCH           = 256                     # Chronos batch_size (tasks per forward pass)
CROSS_LEARNING  = False                   # True shares information across a batch; results then depend on batch makeup
CONTEXT_LENGTH  = None                    # None = the model's own maximum
QUANTILES = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 0.75, 0.8, 0.9, 0.95, 0.99]
# --------------------------------------------------------------------------------------------

assert BACKTEST_CUTOFF + pd.Timedelta(days=HORIZON) <= VAL_END, "the backtest window must end inside the history"
assert BACKTEST_CUTOFF - TRAIN_START > pd.Timedelta(days=365), "need at least a year of context"
print(f"horizon {HORIZON} days | history {N_HIST} days | device {DEVICE}")
print(f"backtest: context to {BACKTEST_CUTOFF.date()}, forecast {(BACKTEST_CUTOFF + pd.Timedelta(days=1)).date()} -> "
      f"{(BACKTEST_CUTOFF + pd.Timedelta(days=HORIZON)).date()}")

## Section 0 — Series cache (built here only if it is missing)
Chronos needs one `.npz` file per series (`train_sales`, `val_sales`) plus `series_table.parquet` and `spend_paths.npy`, all inside `series_cache/`.
The modelling notebook (Step 2) normally writes them. This cell does the same job, so the notebook also works in a folder where Step 2 was never run.

- **Cache found** → it is reused and nothing is rebuilt.
- **Cache missing** → it is built from `local_train_data/chunk_*.parquet` (written by Step 1). This takes a few minutes for ~32,000 series.
- **`REBUILD_CACHE = True`** → rebuild even if a cache exists. Do this after you re-run Step 1 or change any date in `column_roles.json`.

A rebuild deletes only the `.npz` files, `series_table.parquet`, `spend_paths.npy` and `daily_total_sales.npy`. Other files in `series_cache/` (`series_index.parquet`, `shared_cov.pkl`) belong to the TFT notebooks and are left alone.
This cell produces what Chronos needs, nothing more. The TFT prediction notebook also needs `series_index.parquet` and `shared_cov.pkl` from Step 2.


In [ ]:
REBUILD_CACHE = False

local_train_dir   = os.path.join(BASE, "local_train_data")
series_table_path = os.path.join(CACHE_DIR, "series_table.parquet")
spend_paths_path  = os.path.join(CACHE_DIR, "spend_paths.npy")
daily_total_path  = os.path.join(CACHE_DIR, "daily_total_sales.npy")

VAL_SERIES_START  = pd.Timestamp(ROLES["val_series_start"])      # encoder start of the first validation window
static_covariates = list(ROLES["static_covariates"])
series_covariates = list(ROLES["series_covariates"])

def safe_name(key):
    return str(key).replace("<>", "_").replace("/", "_").replace("\\", "_")

_cache_found = os.path.exists(series_table_path) and os.path.exists(spend_paths_path)

if _cache_found and not REBUILD_CACHE:
    print(f"Series cache found in {CACHE_DIR} -- reusing it (REBUILD_CACHE = False).")
else:
    chunk_files = sorted(glob.glob(os.path.join(local_train_dir, "chunk_*.parquet")))
    assert chunk_files, (
        f"No series_cache found in {CACHE_DIR} and no chunk files in {local_train_dir}.\n"
        f"BASE is {BASE}. If your TFT project is in another folder, set BASE (first code cell) to that folder.\n"
        "Otherwise run the data-preparation notebook (Step 1) first.")
    os.makedirs(CACHE_DIR, exist_ok=True)
    for _f in glob.glob(os.path.join(CACHE_DIR, "*.npz")) + [series_table_path, spend_paths_path, daily_total_path]:
        if os.path.exists(_f):
            os.remove(_f)

    rows = []
    daily_total = np.zeros(N_HIST, dtype=np.float64)          # all-series NET_SALES per day, TRAIN_START -> VAL_END
    path_of_digest, path_arrays = {}, []
    for ci, path in enumerate(chunk_files):
        df = pd.read_parquet(path)
        df[time_col] = pd.to_datetime(df[time_col])
        for key, g in df.groupby(group_col, sort=False):
            g = g.sort_values(time_col)
            t = g[time_col].reset_index(drop=True)
            assert (t.diff().dropna() == pd.Timedelta(days=1)).all(), f"{key}: dates are not consecutive"

            row = dict(key=str(key), first_date=t.iloc[0], last_date=t.iloc[-1], n_rows=len(t),
                       n_train=0, n_val=0, total_sales=np.nan, spend_path=-1)
            row.update({c: str(g[c].iloc[0]) for c in static_covariates})
            if len(t) != N_FULL or t.iloc[0] != TRAIN_START or t.iloc[-1] != FORECAST_END:
                rows.append(row)                  # series that does not cover TRAIN_START..FORECAST_END is skipped below
                continue

            is_hist = (t <= VAL_END).to_numpy()
            sales = g[target_col].to_numpy(dtype=np.float32)[is_hist]
            assert np.isfinite(sales).all() and (sales >= 0).all() and np.allclose(sales, np.rint(sales)), \
                f"{key}: non-count target"
            t_h = t[is_hist].reset_index(drop=True)
            tr = (t_h <= TRAIN_END).to_numpy()
            va = (t_h >= VAL_SERIES_START).to_numpy()
            payload = dict(val_sales=sales[va],
                           val_start=np.array(str(t_h[va].iloc[0].date())) if va.any() else np.array(""))
            if tr.any():
                payload.update(train_sales=sales[tr], train_start=np.array(str(t_h[tr].iloc[0].date())))
            np.savez(os.path.join(CACHE_DIR, f"{safe_name(key)}.npz"), **payload)
            daily_total += sales

            total = float(sales.sum())
            pid = -1                                          # series with no sales at all are excluded later
            if total > 0:
                spend = g[series_covariates].to_numpy(dtype=np.float32)      # (N_FULL, 2), already scaled in Step 1
                assert np.isfinite(spend).all(), f"{key}: spend columns contain NaN/inf"
                digest = hashlib.md5(np.round(spend, 4).tobytes()).hexdigest()
                pid = path_of_digest.setdefault(digest, len(path_of_digest))
                if pid == len(path_arrays):
                    path_arrays.append(spend)
            row.update(n_train=int(tr.sum()), n_val=int(va.sum()), total_sales=total, spend_path=int(pid))
            rows.append(row)
        del df
        gc.collect()
        print(f"  chunk {ci+1}/{len(chunk_files)} -- series so far: {len(rows):,} | distinct spend paths so far: {len(path_arrays):,}")

    _built = pd.DataFrame(rows)
    _built.to_parquet(series_table_path, index=False)
    _sp = (np.stack(path_arrays).astype(np.float32) if path_arrays
           else np.zeros((0, N_FULL, len(series_covariates)), dtype=np.float32))
    np.save(spend_paths_path, _sp)
    np.save(daily_total_path, daily_total)
    print(f"Cache built in {CACHE_DIR}: {len(_built):,} series, {len(_sp):,} distinct spend paths.")

# quick consistency check of whichever cache is now in place
_st = pd.read_parquet(series_table_path)
_first = _st.loc[_st["spend_path"] >= 0, "key"].iloc[0]
with np.load(os.path.join(CACHE_DIR, f"{safe_name(_first)}.npz")) as _z:
    assert len(_z["train_sales"]) == (TRAIN_END - TRAIN_START).days + 1, \
        "The cached series do not match the dates in column_roles.json. Set REBUILD_CACHE = True and re-run this cell."
print(f"Cache OK: {len(_st):,} series in series_table.parquet")


## Section 1 — Load the model
The first call downloads the weights from Hugging Face (about 0.5 GB). Chronos-2 reads at most 8,192 days of context; your history is about 1,300 days (the cell below prints the exact number), so all of it is used.

In [ ]:
pipeline = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map=DEVICE)
print("model quantile levels it was trained on:", pipeline.quantiles)
print("model max prediction length:", pipeline.model_prediction_length)
assert pipeline.model_prediction_length >= HORIZON
print(f"history length: {N_HIST:,} days (model context limit 8,192); horizon: {HORIZON} days")

## Section 2 — Load history, spend paths and calendar from the series cache
- Sales history per series = `train_sales` (to `TRAIN_END`) + the last `VAL_END − TRAIN_END` days of `val_sales` (to `VAL_END`). 
- Spend per series comes from `spend_paths.npy` through the series' `spend_path` number (already log-scaled and z-scored in Step 1).
- Two compact festival features are derived from the calendar: `N_ANY` / `D_ANY` (any column of the Navratri / Diwali block is on) and `N_OFFSET` / `D_OFFSET` (signed days from the nearest Navratri / Diwali, clipped to ±45, positive = after). A foundation model reads one numeric "days to Diwali" far more easily than 27 one-day flags.

In [ ]:
def safe_name(k):
    return str(k).replace("<>", "_").replace("/", "_").replace("\\", "_")

series_table = pd.read_parquet(os.path.join(CACHE_DIR, "series_table.parquet"))
series_table = series_table[series_table["total_sales"] > 0].reset_index(drop=True)      # same series the TFT uses
SPEND_PATHS  = np.load(os.path.join(CACHE_DIR, "spend_paths.npy"))                        # (paths, N_FULL, 2)
assert SPEND_PATHS.shape[1] == N_FULL
SPEND_COLS   = list(ROLES["series_covariates"])
STATICS      = list(ROLES["static_covariates"])
print(f"live series: {len(series_table):,} | spend paths: {len(SPEND_PATHS)}")

N_AFTER_TRAIN = (VAL_END - TRAIN_END).days                 # days of val_sales that lie after TRAIN_END
def load_panel(table):
    """Sales history (n, N_HIST) for the series in `table`, plus their spend-path numbers."""
    Y = np.empty((len(table), N_HIST), dtype=np.float32)
    for i, k in enumerate(table["key"]):
        with np.load(os.path.join(CACHE_DIR, f"{safe_name(k)}.npz")) as z:
            tr, va = z["train_sales"], z["val_sales"]
        assert len(tr) == (TRAIN_END - TRAIN_START).days + 1, f"{k}: unexpected train length {len(tr)}"
        Y[i] = np.concatenate([tr, va[-N_AFTER_TRAIN:]])
    return dict(keys=table["key"].to_numpy(), Y=Y, paths=table["spend_path"].to_numpy().astype(int),
                meta=table[STATICS].reset_index(drop=True))

# ---- calendar ------------------------------------------------------------------------------
cal = pd.read_parquet(os.path.join(BASE, ROLES["calendar_path"]))
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.set_index(time_col).loc[DAYS_FULL].astype("float32")
blocks = ROLES["festive_blocks"]
cal["N_ANY"], cal["D_ANY"] = cal[blocks["N"]].max(axis=1), cal[blocks["D"]].max(axis=1)

def signed_offset(dates, flag_days, clip=45):
    """days from each date to the nearest flag day (positive = date is after the festival)."""
    d = (dates.values[:, None] - pd.DatetimeIndex(flag_days).values[None, :]).astype("timedelta64[D]").astype(int)
    nearest = np.abs(d).argmin(axis=1)
    return np.clip(d[np.arange(len(d)), nearest], -clip, clip).astype("float32")

N_DAYS = cal.index[cal["N"] == 1]
D_DAYS = cal.index[cal["D"] == 1]
cal["N_OFFSET"] = signed_offset(cal.index, N_DAYS)
cal["D_OFFSET"] = signed_offset(cal.index, D_DAYS)
print("Navratri days:", [d.date().isoformat() for d in N_DAYS])
print("Diwali days  :", [d.date().isoformat() for d in D_DAYS])

# ---- covariate sets: (calendar columns, include the two spend columns?) ----------------------
_cal_small = [c for c in ["N_ANY", "D_ANY", "N_OFFSET", "D_OFFSET", "IS_TUESDAY", "IS_WEEKEND", "POST_GST_CUT",
                          "DOW_SIN", "DOW_COS", "IS_MONTH_END", "IS_MONTH_START", "DAYS_TO_MONTH_END"] if c in cal.columns]
COV_SETS = {
    "none":            ([], False),
    "calendar":        (_cal_small, False),
    "calendar+spend":  (_cal_small, True),
    "all_flags+spend": (list(ROLES["shared_covariates"]) + ["N_OFFSET", "D_OFFSET"], True),
}
for k in RUN_SETS:
    assert k in COV_SETS, k
    print(f"{k:16s}: {len(COV_SETS[k][0])} calendar columns + {len(SPEND_COLS) if COV_SETS[k][1] else 0} spend columns")

## Section 3 — Build the long-format frames Chronos reads
One row per series per day. `context_df` holds the target and every covariate up to the cutoff; `future_df` holds the same covariates for the 92 forecast days and **no target**. A column present in both is a *known-future* covariate to Chronos; a column only in `context_df` would be past-only.

In [ ]:
def build_frames(panel, rows, cutoff, horizon, shared_cols, use_spend):
    pos = (cutoff - TRAIN_START).days            # index of the last context day
    L, H, n = pos + 1, horizon, len(rows)
    assert L + H <= N_FULL
    ids = np.arange(n)
    ctx = {"item_id": np.repeat(ids, L), "timestamp": np.tile(DAYS_FULL[:L].values, n),
           "target": panel["Y"][rows, :L].ravel()}
    fut = {"item_id": np.repeat(ids, H), "timestamp": np.tile(DAYS_FULL[L:L + H].values, n)}
    for c in shared_cols:
        v = cal[c].to_numpy(np.float32)
        ctx[c], fut[c] = np.tile(v[:L], n), np.tile(v[L:L + H], n)
    if use_spend:
        sp = SPEND_PATHS[panel["paths"][rows]]                      # (n, N_FULL, 2)
        assert (panel["paths"][rows] >= 0).all(), "a series has no spend path"
        for j, c in enumerate(SPEND_COLS):
            ctx[c], fut[c] = sp[:, :L, j].ravel(), sp[:, L:L + H, j].ravel()
    has_cov = bool(shared_cols) or use_spend
    return pd.DataFrame(ctx), (pd.DataFrame(fut) if has_cov else None)


def repair(q):
    """quantiles must not be negative (counts) and must not cross."""
    return np.maximum.accumulate(np.clip(q, 0, None), axis=-1)


def chronos_forecast(panel, rows, cutoff, horizon, shared_cols, use_spend, label=""):
    """Returns repaired quantile forecasts, shape (len(rows), horizon, len(QUANTILES))."""
    rows = np.asarray(rows)
    out = np.empty((len(rows), horizon, len(QUANTILES)), dtype=np.float32)
    t0 = time.time()
    for b in range(0, len(rows), BLOCK):
        r = rows[b:b + BLOCK]
        ctx, fut = build_frames(panel, r, cutoff, horizon, shared_cols, use_spend)
        pred = pipeline.predict_df(
            ctx, future_df=fut, id_column="item_id", timestamp_column="timestamp", target="target",
            prediction_length=horizon, quantile_levels=QUANTILES, batch_size=BATCH,
            context_length=CONTEXT_LENGTH, cross_learning=CROSS_LEARNING, freq="D")
        pred = pred.sort_values(["item_id", "timestamp"])
        assert len(pred) == len(r) * horizon, f"expected {len(r) * horizon} rows, got {len(pred)}"
        assert (pred["item_id"].to_numpy().reshape(len(r), horizon)[:, 0] == np.arange(len(r))).all(), "series order changed"
        assert pred["timestamp"].iloc[0] == cutoff + pd.Timedelta(days=1), "forecast starts on the wrong day"
        q = pred[[str(l) for l in QUANTILES]].to_numpy(np.float32).reshape(len(r), horizon, len(QUANTILES))
        assert np.isfinite(q).all(), "NaN/inf in the forecast"
        out[b:b + len(r)] = q
        el = time.time() - t0
        print(f"  {label} {b + len(r):,}/{len(rows):,} series | {(b + len(r)) / el:,.0f} series/s", end="\r")
    print()
    return repair(out)

## Section 4 — From quantiles to a mean
The metrics and your PRED_MEAN need a **mean**, but Chronos gives quantiles (`predictions` is the 0.5 quantile). The mean of a distribution is the average of its quantile function over (0, 1), so each quantile level is weighted by the stretch of probability nearest to it. With 16 levels from 0.01 to 0.99 this is a good approximation unless the distribution has a heavy upper tail beyond the 99th percentile, in which case it *under*-states the mean. Treat `chronos_mean` as a lower-biased estimate on spiky series.

In [ ]:
def grid_mean(q, levels=QUANTILES):
    lv = np.asarray(levels)
    edges = np.concatenate([[0.0], (lv[1:] + lv[:-1]) / 2, [1.0]])
    return (q * np.diff(edges)).sum(-1)

def level(q, p):
    return q[..., QUANTILES.index(p)]

# quick check on a known distribution: a uniform(0, 10) variable has mean 5
_u = np.array([10 * l for l in QUANTILES])[None, None, :]
print("grid_mean check, Uniform(0,10) -> expect about 5:", float(grid_mean(_u)[0, 0]))

## Section 5 — Baselines
- **`fasn`** (festival-aligned seasonal naive): forecast for day *d* = sales on day *d − k*, where *k* = this year's Navratri date minus last year's (about 354 days when the festival moved 11 days earlier). Navratri and Diwali are 28 days apart in every year in your calendar, so one shift aligns both festivals. The cell prints both shifts and warns if they differ.
- **`trailing`**: the mean of the last 91 days of context, repeated.
If Chronos cannot beat these, the foundation model is adding nothing.

In [ ]:
def fasn_shift(cutoff):
    year = (cutoff + pd.Timedelta(days=1)).year
    def pick(days, y):
        c = [d for d in days if d.year == y]
        assert len(c) == 1, f"expected one festival day in {y}, found {c}"
        return c[0]
    kn = (pick(N_DAYS, year) - pick(N_DAYS, year - 1)).days
    kd = (pick(D_DAYS, year) - pick(D_DAYS, year - 1)).days
    if abs(kn - kd) > 3:
        print(f"*** WARNING: Navratri shifts by {kn} days but Diwali by {kd}; the aligned baseline fits only one of them")
    print(f"lag back to the same festival last year: Navratri {kn} days, Diwali {kd} days (using {kn})")
    return kn

def baselines(panel, rows, cutoff, horizon):
    pos = (cutoff - TRAIN_START).days
    k = fasn_shift(cutoff)
    src = np.arange(pos + 1, pos + 1 + horizon) - k          # day indices to copy from
    assert src.max() <= pos and src.min() >= 0, "the shifted window leaves the available context"
    Y = panel["Y"][rows]
    return {"fasn": Y[:, src].astype(np.float32),
            "trailing": np.repeat(Y[:, pos - 90:pos + 1].mean(1, keepdims=True), horizon, axis=1).astype(np.float32)}

## Section 6 — Backtest: cutoff 2025-09-30, forecast Oct–Dec 2025
Score every method at several levels. A forecast that is poor per series-day but good for family or national totals may still be useful for your SOQ; one that is poor at totals is not.

| Metric | Meaning |
|---|---|
| `WAPE series-day` | Σ|y − ŷ| / Σy over every series and day (lower is better) |
| `WAPE family-day`, `dealer-week`, `national-day` | same after summing to that level first |
| `bias %` | (Σŷ − Σy) / Σy over the whole window |
| `WQL` | weighted quantile loss, Chronos only: 2·Σ pinball / Σy, averaged over the quantile levels |
| `Oct / Nov / Dec %` | share of the window total in each month, to compare with the actual shape |

In [ ]:
def _group_sum(arr, labels):
    return pd.DataFrame(arr).groupby(np.asarray(labels)).sum().to_numpy()

def wape(y, f):
    return float(np.abs(y - f).sum() / max(float(y.sum()), 1e-9))

def pinball_wql(y, q):
    tot = 0.0
    for i, tau in enumerate(QUANTILES):
        d = y - q[..., i]
        tot += np.maximum(tau * d, (tau - 1) * d).sum()
    return float(2 * tot / len(QUANTILES) / max(float(y.sum()), 1e-9))

def score_all(y, preds, meta, dates, qpred=None):
    n, H = y.shape
    months = pd.DatetimeIndex(dates).month
    rows = []
    for name, f in preds.items():
        fam = _group_sum(f, meta["MODEL_FAMILY"]);  ya = _group_sum(y, meta["MODEL_FAMILY"])
        wk = (H // 7) * 7
        dw  = _group_sum(f[:, :wk].reshape(n, -1, 7).sum(2), meta["PARENT_DEALER_CODE"])
        ydw = _group_sum(y[:, :wk].reshape(n, -1, 7).sum(2), meta["PARENT_DEALER_CODE"])
        r = {"method": name, "WAPE series-day": wape(y, f), "WAPE family-day": wape(ya, fam),
             "WAPE dealer-week": wape(ydw, dw), "WAPE national-day": wape(y.sum(0), f.sum(0)),
             "bias %": 100 * (float(f.sum()) / max(float(y.sum()), 1e-9) - 1),
             "WQL": pinball_wql(y, qpred[name]) if (qpred is not None and name in qpred) else np.nan}
        for m in sorted(set(months)):
            r[f"{pd.Timestamp(2000, m, 1).strftime('%b')} %"] = 100 * float(f[:, months == m].sum()) / max(float(f.sum()), 1e-9)
        rows.append(r)
    ya_m = {m: float(y[:, months == m].sum()) / max(float(y.sum()), 1e-9) * 100 for m in sorted(set(months))}
    rows.append({"method": "ACTUAL", **{f"{pd.Timestamp(2000, m, 1).strftime('%b')} %": v for m, v in ya_m.items()}})
    return pd.DataFrame(rows).set_index("method")

# ---- the sample and its history ----------------------------------------------------------------
rng = np.random.default_rng(SEED)
if SAMPLE_N is None or SAMPLE_N >= len(series_table):
    bt_table = series_table
else:
    bt_table = series_table.iloc[np.sort(rng.choice(len(series_table), SAMPLE_N, replace=False))].reset_index(drop=True)
bt = load_panel(bt_table)
rows_all = np.arange(len(bt["keys"]))
pos_c = (BACKTEST_CUTOFF - TRAIN_START).days
y_true = bt["Y"][:, pos_c + 1: pos_c + 1 + HORIZON]
bt_dates = DAYS_FULL[pos_c + 1: pos_c + 1 + HORIZON]
print(f"backtest series: {len(rows_all):,} | actual units in window: {y_true.sum():,.0f} | "
      f"share of series-days with sales > 0: {(y_true > 0).mean() * 100:.1f}%")
base = baselines(bt, rows_all, BACKTEST_CUTOFF, HORIZON)

In [ ]:
BT = {}                                       # results per covariate set, kept for the plots below
tables = {}
for name in RUN_SETS:
    shared_cols, use_spend = COV_SETS[name]
    q = chronos_forecast(bt, rows_all, BACKTEST_CUTOFF, HORIZON, shared_cols, use_spend, label=name)
    BT[name] = q
    preds = {"chronos_mean": grid_mean(q), "chronos_median": level(q, 0.5), **base}
    tables[name] = score_all(y_true, preds, bt["meta"], bt_dates, qpred={"chronos_mean": q, "chronos_median": q})

pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
for name, t in tables.items():
    print(f"\n===== Chronos-2 with covariate set: {name} =====")
    print(t.round(3).to_string())

**How to read the table.** Compare the `chronos_mean` rows across covariate sets first: the gap between `none` and `calendar` is what festival information is worth to Chronos; `calendar` vs `calendar+spend` is what spend is worth. Then compare the best Chronos row with `fasn`. If `fasn` wins at `national-day` and `dealer-week`, the foundation model is not earning its keep for aggregate planning. `WQL` is only for Chronos (the baselines are point forecasts).

Check the month shares against the `ACTUAL` row: your TFT's problem was shape (November far too low, December far too high). A model that cannot put the peak in November gets little credit for a good total.

In [ ]:
best = min(tables, key=lambda k: tables[k].loc["chronos_mean", "WAPE national-day"])
q = BT[best]
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(bt_dates, y_true.sum(0), label="actual", color="black", lw=2)
ax.plot(bt_dates, grid_mean(q).sum(0), label=f"chronos mean ({best})")
ax.plot(bt_dates, base["fasn"].sum(0), label="festival-aligned naive", ls="--")
ax.plot(bt_dates, base["trailing"].sum(0), label="trailing mean", ls=":")
ax.set_title(f"National daily units in the sample, Oct–Dec {bt_dates[0].year}"); ax.legend(); plt.tight_layout(); plt.show()

## Section 7 — Live forecast, Oct–Dec 2026 (all series)
Context ends 2026-09-30, covariates are the calendar plus the **planned** spend already in your cache. Writes a parquet in the same layout as your Step 3 output (`PRED_MEAN`, `PRED_Q45`, `PRED_Q55`, `PRED_Q70`, `PRED_Q75`), so your Step 3 shape-check cell can read it unchanged.
Set `LIVE_SET` to the covariate set that won Section 6. This runs on every series; expect minutes, not seconds.

In [ ]:
LIVE_SET = "calendar+spend"                     # <- set from the Section 6 result
RUN_LIVE = False                                # flip to True when you want the full run

if RUN_LIVE:
    live = load_panel(series_table)
    shared_cols, use_spend = COV_SETS[LIVE_SET]
    q = chronos_forecast(live, np.arange(len(live["keys"])), VAL_END, HORIZON, shared_cols, use_spend, label=LIVE_SET)
    fdates = pd.date_range(FORECAST_START, FORECAST_END, freq="D")
    n = len(live["keys"])
    out = pd.DataFrame({group_col: np.repeat(live["keys"], HORIZON), time_col: np.tile(fdates.values, n),
                        "PRED_MEAN": grid_mean(q).ravel()})
    for p in (0.45, 0.55, 0.7, 0.75):
        out[f"PRED_Q{round(100 * p)}"] = level(q, p).ravel()
    out_dir = os.path.join(BASE, "chronos_predictions"); os.makedirs(out_dir, exist_ok=True)
    CHRONOS_LIVE_PATH = os.path.join(out_dir, f"chronos2_{LIVE_SET.replace('+', '_')}_{pd.Timestamp.now():%Y%m%d_%H%M%S}.parquet")
    assert len(out) == n * HORIZON and not out.duplicated([group_col, time_col]).any()
    out.to_parquet(CHRONOS_LIVE_PATH, index=False)
    out["MONTH"] = out[time_col].dt.month_name()
    print("written ->", CHRONOS_LIVE_PATH)
    print(out.groupby("MONTH")["PRED_MEAN"].sum().round(0).to_string())

## Section 8 — Fair head-to-head with the TFT on real Oct–Nov 2026 sales
Today there are real sales for the start of the window, including Diwali (8 Nov). Neither model has seen them, so this is the one comparison that is fair to both.

**You need to supply the actuals.** Export from Snowflake (same series key as everywhere else) and save as parquet:
```sql
SELECT REPLACE(PARENT_DEALER_CODE_MODEL_FAMILY, '<>', '_') AS PARENT_DEALER_CODE_MODEL_FAMILY,
       CAL_DATE, NET_SALES
FROM   MOP_DATABASE.SOQ.TFT_DAILY_DATA_FOR_FORECASTING_WITH_XGB_FEATURES
WHERE  CAL_DATE BETWEEN '2026-10-01' AND '<last day with complete sales>'
```
Use only days whose sales are complete; a half-loaded last day makes every model look like it over-forecasts. The cell scores only the dates present in the actuals file.

In [ ]:
ACTUALS_PATH      = os.path.join(BASE, "actuals_oct_nov_2026.parquet")
TFT_PRED_PATH     = r""            # the Step 3 parquet, e.g. predictions_2026_negbin_torch_convention/<run>/<model>_predictions.parquet
CHRONOS_PRED_PATH = globals().get("CHRONOS_LIVE_PATH", r"")

paths_ok = all(p and os.path.exists(p) for p in (ACTUALS_PATH, TFT_PRED_PATH, CHRONOS_PRED_PATH))
if not paths_ok:
    print("Skipped: set ACTUALS_PATH, TFT_PRED_PATH and run Section 7 (RUN_LIVE=True) first.")
else:
    act = pd.read_parquet(ACTUALS_PATH, columns=[group_col, time_col, target_col]); act[time_col] = pd.to_datetime(act[time_col])
    tft = pd.read_parquet(TFT_PRED_PATH, columns=[group_col, time_col, "PRED_MEAN"]).rename(columns={"PRED_MEAN": "tft"})
    chr_ = pd.read_parquet(CHRONOS_PRED_PATH, columns=[group_col, time_col, "PRED_MEAN"]).rename(columns={"PRED_MEAN": "chronos"})
    for d in (tft, chr_): d[time_col] = pd.to_datetime(d[time_col])
    # only dates present in the actuals, and only series both models forecast
    m = act.merge(tft, on=[group_col, time_col], how="inner").merge(chr_, on=[group_col, time_col], how="inner")
    print(f"scored dates: {m[time_col].min().date()} -> {m[time_col].max().date()} | series: {m[group_col].nunique():,} | "
          f"units: {m[target_col].sum():,.0f}")
    meta = series_table.set_index("key")[STATICS]
    m = m.join(meta, on=group_col)
    rows = []
    for name in ("tft", "chronos"):
        f, y = m[name].to_numpy(), m[target_col].to_numpy()
        fam = m.groupby(["MODEL_FAMILY", time_col])[[name, target_col]].sum()
        dw  = m.assign(week=m[time_col].dt.to_period("W")).groupby(["PARENT_DEALER_CODE", "week"])[[name, target_col]].sum()
        nat = m.groupby(time_col)[[name, target_col]].sum()
        rows.append({"model": name, "WAPE series-day": wape(y, f), "WAPE family-day": wape(fam[target_col].to_numpy(), fam[name].to_numpy()),
                     "WAPE dealer-week": wape(dw[target_col].to_numpy(), dw[name].to_numpy()),
                     "WAPE national-day": wape(nat[target_col].to_numpy(), nat[name].to_numpy()),
                     "bias %": 100 * (f.sum() / y.sum() - 1)})
    print(pd.DataFrame(rows).set_index("model").round(3).to_string())